In [1]:
import numpy as np
import tensorflow as tf
import shutil
import os 
import random

In [2]:
folder_gambar = './DatasetRiceDiseaseProject/data'

In [3]:
split_dataset = './split_dataset'

In [4]:
#cek jumlah data 
def cek_jumlahdata(data):
    # hitung total semua file
    total_file = sum(
        len([f for f in os.listdir(os.path.join(data, d)) 
             if os.path.isfile(os.path.join(data, d, f))])
        for d in os.listdir(data) if os.path.isdir(os.path.join(data, d))
    )

    # tampilkan jumlah tiap kelas + persentase
    for nama_kelas in os.listdir(data):
        path_kelas = os.path.join(data, nama_kelas)
        if os.path.isdir(path_kelas):
            jumlah_file = len([f for f in os.listdir(path_kelas) if os.path.isfile(os.path.join(path_kelas, f))])
            print(f"Kelas: {nama_kelas}, Jumlah gambar: {jumlah_file}, Persentase: {jumlah_file/total_file:.2%}" )
    print(f"Total jumlah data gambar:{total_file}")

In [5]:
#cek jumlah data train di tiap penyakit
cek_jumlahdata(folder_gambar)

Kelas: BacterialLeafBlight, Jumlah gambar: 3010, Persentase: 16.10%
Kelas: BrownSpot, Jumlah gambar: 3643, Persentase: 19.49%
Kelas: Healthy, Jumlah gambar: 1488, Persentase: 7.96%
Kelas: Hispa, Jumlah gambar: 2026, Persentase: 10.84%
Kelas: Leaf_Blast, Jumlah gambar: 2219, Persentase: 11.87%
Kelas: Leaf_scald, Jumlah gambar: 1670, Persentase: 8.93%
Kelas: Neck_blast, Jumlah gambar: 1322, Persentase: 7.07%
Kelas: Sheath_blight, Jumlah gambar: 1578, Persentase: 8.44%
Kelas: Tungro, Jumlah gambar: 1740, Persentase: 9.31%
Total jumlah data gambar:18696


In [4]:
import splitfolders
splitfolders.ratio(folder_gambar, output="split_dataset", seed=42, ratio=(.7, .15, .15))

In [9]:
train = './split_dataset/train'
test = './split_dataset/test'
val = './split_dataset/val'

In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Inisialisasi data generator train dan test set
train_datagen = ImageDataGenerator(
      rescale = 1./255,
      rotation_range=30,
      width_shift_range=0.2,
      height_shift_range=0.2,
      shear_range=0.2,
      zoom_range=0.2,
      horizontal_flip=True,
      vertical_flip=True)

test_datagen = ImageDataGenerator(
                    rescale=1./255)

val_datagen = ImageDataGenerator(
                    rescale=1./255)


In [11]:
    train_generator = train_datagen.flow_from_directory(
        train,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
        subset='training'
    )

    validation_generator = val_datagen.flow_from_directory(
        val,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
    )

    test_generator = test_datagen.flow_from_directory(
        test,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
    )



Found 13085 images belonging to 9 classes.
Found 2800 images belonging to 9 classes.
Found 2811 images belonging to 9 classes.


In [ ]:
print(train_generator.class_indices)
print(validation_generator.class_indices)

{'BacterialLeafBlight': 0, 'BrownSpot': 1, 'Healthy': 2, 'Hispa': 3, 'Leaf_Blast': 4, 'Leaf_scald': 5, 'Neck_blast': 6, 'Sheath_blight': 7, 'Tungro': 8}
{'BacterialLeafBlight': 0, 'BrownSpot': 1, 'Healthy': 2, 'Hispa': 3, 'Leaf_Blast': 4, 'Leaf_scald': 5, 'Neck_blast': 6, 'Sheath_blight': 7, 'Tungro': 8}


In [8]:
base_model = tf.keras.applications.Xception(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

In [9]:
base_model.trainable=False

In [11]:
#for layer in base_model.layers:
    #layer.trainable = False

In [ ]:
from tensorflow.keras.layers import Flatten,Dense,BatchNormalization
from tensorflow.keras import optimizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

In [12]:

model = Sequential([
    base_model,
    Flatten(),
   Dense(256,activation='relu'),
   BatchNormalization(),
   Dense(9,activation='softmax'),
])

In [13]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 xception (Functional)       (None, 7, 7, 2048)        20861480  
                                                                 
 flatten (Flatten)           (None, 100352)            0         
                                                                 
 dense (Dense)               (None, 256)               25690368  
                                                                 
 batch_normalization_4 (Batc  (None, 256)              1024      
 hNormalization)                                                 
                                                                 
 dense_1 (Dense)             (None, 9)                 2313      
                                                                 
Total params: 46,555,185
Trainable params: 25,693,193
Non-trainable params: 20,861,992
___________________________________

In [24]:
lr_rate = 0.0001
num_epochs = 10

In [27]:
model.compile(loss='categorical_crossentropy',
              optimizer=optimizers.Adam(learning_rate=lr_rate),
              metrics=['accuracy'],)

In [28]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5)

In [38]:
history = model.fit(
    train_generator,
    epochs=num_epochs,
    validation_data=validation_generator,
    callbacks=[early_stopping]
)

Epoch 1/10
409/409 [==============================] - 703s 2s/step - loss: 0.7543 - accuracy: 0.7561 - val_loss: 0.5527 - val_accuracy: 0.8161
Epoch 2/10
409/409 [==============================] - 662s 2s/step - loss: 0.5057 - accuracy: 0.8261 - val_loss: 0.4722 - val_accuracy: 0.8386
Epoch 3/10
409/409 [==============================] - 674s 2s/step - loss: 0.4451 - accuracy: 0.8446 - val_loss: 0.3814 - val_accuracy: 0.8711
Epoch 4/10
409/409 [==============================] - 657s 2s/step - loss: 0.4060 - accuracy: 0.8557 - val_loss: 0.3465 - val_accuracy: 0.8821
Epoch 5/10
409/409 [==============================] - 649s 2s/step - loss: 0.3812 - accuracy: 0.8655 - val_loss: 0.3262 - val_accuracy: 0.8764
Epoch 6/10
409/409 [==============================] - 659s 2s/step - loss: 0.3611 - accuracy: 0.8715 - val_loss: 0.3390 - val_accuracy: 0.8782
Epoch 7/10
409/409 [==============================] - 1330s 3s/step - loss: 0.3585 - accuracy: 0.8695 - val_loss: 0.2985 - val_accuracy: 0.892

In [39]:
model.save('computer_vision.keras')

In [7]:
from keras.models import load_model
model = load_model('computer_vision.keras')

In [19]:
history2 = model.fit(
    train_generator,
    epochs=num_epochs,
    validation_data=validation_generator,
    callbacks=[early_stopping]
)

Epoch 1/10
409/409 [==============================] - 1056s 3s/step - loss: 0.3095 - accuracy: 0.8883 - val_loss: 0.2826 - val_accuracy: 0.8996
Epoch 2/10
409/409 [==============================] - 1345s 3s/step - loss: 0.3091 - accuracy: 0.8880 - val_loss: 0.2620 - val_accuracy: 0.9050
Epoch 3/10
409/409 [==============================] - 855s 2s/step - loss: 0.2943 - accuracy: 0.8938 - val_loss: 0.2538 - val_accuracy: 0.9089
Epoch 4/10
409/409 [==============================] - 688s 2s/step - loss: 0.2954 - accuracy: 0.8945 - val_loss: 0.2556 - val_accuracy: 0.9086
Epoch 5/10
409/409 [==============================] - 684s 2s/step - loss: 0.2946 - accuracy: 0.8951 - val_loss: 0.2865 - val_accuracy: 0.8982
Epoch 6/10
409/409 [==============================] - 686s 2s/step - loss: 0.2784 - accuracy: 0.8979 - val_loss: 0.2658 - val_accuracy: 0.9064
Epoch 7/10
409/409 [==============================] - 670s 2s/step - loss: 0.2846 - accuracy: 0.8982 - val_loss: 0.2536 - val_accuracy: 0.90

In [21]:
history3 = model.fit(
    train_generator,
    epochs=num_epochs,
    validation_data=validation_generator,
    callbacks=[early_stopping]
)

Epoch 1/10
409/409 [==============================] - 686s 2s/step - loss: 0.2595 - accuracy: 0.9019 - val_loss: 0.2426 - val_accuracy: 0.9100
Epoch 2/10
409/409 [==============================] - 702s 2s/step - loss: 0.2616 - accuracy: 0.9068 - val_loss: 0.2482 - val_accuracy: 0.9096
Epoch 3/10
409/409 [==============================] - 715s 2s/step - loss: 0.2593 - accuracy: 0.9055 - val_loss: 0.2607 - val_accuracy: 0.9068
Epoch 4/10
409/409 [==============================] - 664s 2s/step - loss: 0.2589 - accuracy: 0.9048 - val_loss: 0.2283 - val_accuracy: 0.9182
Epoch 5/10
409/409 [==============================] - 688s 2s/step - loss: 0.2484 - accuracy: 0.9097 - val_loss: 0.2514 - val_accuracy: 0.9068
Epoch 6/10
409/409 [==============================] - 702s 2s/step - loss: 0.2498 - accuracy: 0.9089 - val_loss: 0.2383 - val_accuracy: 0.9143
Epoch 7/10
409/409 [==============================] - 661s 2s/step - loss: 0.2474 - accuracy: 0.9081 - val_loss: 0.2331 - val_accuracy: 0.9139

In [22]:
model.save('computer_visionv2.keras')

In [26]:
model.evaluate(test_generator)

88/88 [==============================] - 85s 963ms/step - loss: 0.2225 - accuracy: 0.9182


[0.2224789261817932, 0.9181785583496094]

In [ ]:
from keras.models import load_model
model = load_model('computer_visionv2.keras')

In [13]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = './brownspot3-crop2.jpg'
img = image.load_img(img_path, target_size=(224, 224))

In [ ]:
#predict_image function
def prediksi_gambar(img,model):
  #Ubah gambar ke array
  img_array = image.img_to_array(img)
  img_array = np.expand_dims(img_array, axis=0)
  img_array /= 255.0

  #Prediksi
  prediction = model.predict(img_array)
  
  #Ubah hasil prediksi dari bentuk array ke nama kelas
  class_labels = list(train_generator.class_indices.keys())
  predicted_class_index = np.argmax(prediction)
  predicted_class_name = class_labels[predicted_class_index]

  print(f'Prediksi model: {predicted_class_name}')



In [23]:
prediksi_gambar(img,model)

1/1 [==============================] - 2s 2s/step
Prediksi model: BrownSpot


In [29]:
history4 = model.fit(
    train_generator,
    epochs=num_epochs,
    validation_data=validation_generator,
    callbacks=[early_stopping]
)

Epoch 1/10
409/409 [==============================] - 717s 2s/step - loss: 0.2379 - accuracy: 0.9147 - val_loss: 0.2195 - val_accuracy: 0.9175
Epoch 2/10
409/409 [==============================] - 705s 2s/step - loss: 0.2321 - accuracy: 0.9152 - val_loss: 0.2209 - val_accuracy: 0.9214
Epoch 3/10
409/409 [==============================] - 704s 2s/step - loss: 0.2399 - accuracy: 0.9125 - val_loss: 0.2368 - val_accuracy: 0.9182
Epoch 4/10
409/409 [==============================] - 703s 2s/step - loss: 0.2311 - accuracy: 0.9143 - val_loss: 0.2365 - val_accuracy: 0.9136
Epoch 5/10
409/409 [==============================] - 713s 2s/step - loss: 0.2295 - accuracy: 0.9166 - val_loss: 0.2122 - val_accuracy: 0.9207
Epoch 6/10
409/409 [==============================] - 713s 2s/step - loss: 0.2295 - accuracy: 0.9170 - val_loss: 0.2154 - val_accuracy: 0.9200
Epoch 7/10
409/409 [==============================] - 716s 2s/step - loss: 0.2257 - accuracy: 0.9187 - val_loss: 0.2057 - val_accuracy: 0.9250

In [ ]:
model.save('computer_visionv3.keras')

In [12]:
from keras.models import load_model
model = load_model('computer_visionv3.keras')

In [16]:
prediksi_gambar(img,model)

1/1 [==============================] - 40s 40s/step
Prediksi model: BrownSpot


In [24]:
class_indices = train_generator.class_indices
class_indices 

{'BacterialLeafBlight': 0,
 'BrownSpot': 1,
 'Healthy': 2,
 'Hispa': 3,
 'Leaf_Blast': 4,
 'Leaf_scald': 5,
 'Neck_blast': 6,
 'Sheath_blight': 7,
 'Tungro': 8}

In [ ]:
#buat label dalam bentuk labels.txt 
num_classes = len(class_indices)
labels = [None] * num_classes
for name, idx in train_generator.class_indices.items():
    labels[idx] = name

with open("labels.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(labels))

In [5]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Simpan hasilnya
with open("model_padi.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmp12fkc4tz\assets


INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmp12fkc4tz\assets
